# Back-Face Consistency Fine-Tuning v2 for TripoSR

## Improvements over v1
1. **Silhouette loss** — prevents density collapse during fine-tuning
2. **Discriminative learning rates** — backbone at 100x lower LR than decoder
3. **Back-face loss warm-up** — linearly ramp over first 3 epochs
4. **Lower back-face weight** — 0.05 vs 0.1
5. **Train on both hard + easy sets** — v1 only trained on hard
6. **Additional metrics** — PSNR and LPIPS alongside CD and F-score

### Before running:
1. Go to **Runtime → Change runtime type → GPU (A100 recommended) → Save**
2. Run cells in order

---

## Cell 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 2 — Clone repos and set up project directory

In [ ]:
import os

BASE_DIR = '/content/drive/MyDrive/backface_v2'
TRIPOSR_DIR = os.path.join(BASE_DIR, 'TripoSR')
SCRIPTS_DIR = os.path.join(BASE_DIR, 'project_scripts')

os.makedirs(BASE_DIR, exist_ok=True)

if not os.path.exists(TRIPOSR_DIR):
    !git clone https://github.com/VAST-AI-Research/TripoSR.git {TRIPOSR_DIR}
    print('TripoSR cloned.')
else:
    print('TripoSR already exists, skipping clone.')

if not os.path.exists(SCRIPTS_DIR):
    !git clone https://github.com/vvalligv/CSCI_566_2Dto3D.git {SCRIPTS_DIR}
    print('Project repo cloned.')
else:
    print('Project repo already exists, skipping clone.')

print(f'\nProject directory: {BASE_DIR}')
!ls {BASE_DIR}

## Cell 3 — Install dependencies

Pin `numpy<2.0` first to avoid compatibility issues with trimesh/TripoSR.

In [ ]:
!pip install 'numpy<2.0' -q
!pip install -r {TRIPOSR_DIR}/requirements.txt -q
!pip install trimesh rtree objaverse PyMCubes lpips scikit-image -q
!pip install 'numpy<2.0' -q  # re-pin after dependency solver

import numpy as np
print(f'numpy: {np.__version__}')

import torch
print(f'torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}')

## Cell 4 — Patch TripoSR

Patches:
- `isosurface.py`: Use PyMCubes instead of torchmcubes (which won't build on Colab)
- `nerf_renderer.py`: Return `(rgb, opacity)` so the training script can use silhouette loss
- `system.py`: Unpack the new renderer return format

In [ ]:
import textwrap

# --- Patch isosurface.py: use PyMCubes ---
isosurface_path = os.path.join(TRIPOSR_DIR, 'tsr/models/isosurface.py')
with open(isosurface_path) as f:
    content = f.read()

if 'mcubes' not in content:
    old_import = '''try:
    from torchmcubes import marching_cubes
except ImportError:
    from skimage.measure import marching_cubes as skimage_mc
    import torch
    def marching_cubes(vol, level):
        vol_np = vol.cpu().numpy()
        verts, faces, _, _ = skimage_mc(vol_np, level=float(level))
        return torch.from_numpy(verts.copy()).float(), torch.from_numpy(faces.copy().astype(\'int64\'))'''

    # Write the entire patched file
    patched = '''from typing import Callable, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn

try:
    from torchmcubes import marching_cubes
except ImportError:
    try:
        import mcubes
        def marching_cubes(vol, level):
            vol_np = vol.cpu().numpy().astype(np.float64)
            verts, faces = mcubes.marching_cubes(vol_np, float(level))
            return (torch.from_numpy(verts.copy()).float(),
                    torch.from_numpy(faces.copy().astype("int64")))
    except ImportError:
        from skimage.measure import marching_cubes as skimage_mc
        def marching_cubes(vol, level):
            vol_np = vol.cpu().numpy()
            lo, hi = float(vol_np.min()), float(vol_np.max())
            lv = float(level)
            if lv < lo:
                lv = lo + 1e-6 * (hi - lo) if hi > lo else lo
            elif lv > hi:
                lv = hi - 1e-6 * (hi - lo) if hi > lo else hi
            verts, faces, _, _ = skimage_mc(vol_np, level=lv)
            return (torch.from_numpy(verts.copy()).float(),
                    torch.from_numpy(faces.copy().astype("int64")))
'''
    # Read rest of file after imports
    with open(isosurface_path) as f:
        lines = f.readlines()
    # Find where the class definition starts
    class_idx = next(i for i, l in enumerate(lines) if 'class IsosurfaceHelper' in l)
    patched += '\n' + ''.join(lines[class_idx:])
    with open(isosurface_path, 'w') as f:
        f.write(patched)
    print('Patched isosurface.py (PyMCubes fallback)')
else:
    print('isosurface.py already patched.')

# --- Patch nerf_renderer.py: return (rgb, opacity) ---
renderer_path = os.path.join(TRIPOSR_DIR, 'tsr/models/nerf_renderer.py')
with open(renderer_path) as f:
    rcontent = f.read()

if 'comp_rgb, opacity' not in rcontent:
    rcontent = rcontent.replace(
        'comp_rgb = (weights[..., None] * colors).sum(dim=-2)',
        'comp_rgb = (weights[..., None] * colors).sum(dim=-2)\n'
        '        opacity = weights.sum(dim=-1)'
    )
    rcontent = rcontent.replace(
        'return comp_rgb',
        'return comp_rgb, opacity'
    )
    # Fix the forward method that stacks results
    rcontent = rcontent.replace(
        'out = self._forward(decoder, triplane, rays_o_, rays_d_)\n            comp_rgb.append(out)',
        'rgb_, opa_ = self._forward(decoder, triplane, rays_o_, rays_d_)\n'
        '            comp_rgb.append(rgb_)\n'
        '            comp_opacity.append(opa_)'
    )
    rcontent = rcontent.replace(
        'comp_rgb = []\n',
        'comp_rgb = []\n        comp_opacity = []\n'
    )
    rcontent = rcontent.replace(
        'comp_rgb = torch.cat(comp_rgb, dim=0)',
        'comp_rgb = torch.cat(comp_rgb, dim=0)\n'
        '        comp_opacity = torch.cat(comp_opacity, dim=0)'
    )
    rcontent = rcontent.replace(
        'return comp_rgb.view(*prefix, H, W, -1)',
        'return comp_rgb.view(*prefix, H, W, -1), comp_opacity.view(*prefix, H, W)'
    )
    with open(renderer_path, 'w') as f:
        f.write(rcontent)
    print('Patched nerf_renderer.py (returns rgb + opacity)')
else:
    print('nerf_renderer.py already patched.')

# --- Patch system.py: unpack (image, opacity) ---
system_path = os.path.join(TRIPOSR_DIR, 'tsr/system.py')
with open(system_path) as f:
    scontent = f.read()

if 'image, _opacity' not in scontent:
    scontent = scontent.replace(
        'image = self.renderer(',
        'image, _opacity = self.renderer('
    )
    with open(system_path, 'w') as f:
        f.write(scontent)
    print('Patched system.py (unpack renderer output)')
else:
    print('system.py already patched.')

print('\nAll patches applied.')

## Cell 5 — Download Objaverse GLBs and render dataset images

Downloads the 20 3D objects from Objaverse and renders front-view images using ray casting (no OpenGL/display needed).

In [ ]:
import sys
sys.path.insert(0, TRIPOSR_DIR)

import json
import objaverse
import trimesh
import numpy as np
from PIL import Image

DATA_DIR = os.path.join(SCRIPTS_DIR, 'project_scripts')
SPLIT_JSON = os.path.join(DATA_DIR, 'dataset_split.json')
RENDER_DIR = os.path.join(BASE_DIR, 'dataset_renders')
RENDER_MANIFEST = os.path.join(DATA_DIR, 'render_manifest.json')

# Download GLBs
with open(SPLIT_JSON) as f:
    split = json.load(f)

all_uids = list({obj['uid'] for obj in split['hard_set'] + split['easy_set']})
print(f'Downloading {len(all_uids)} Objaverse objects...')
objects = objaverse.load_objects(uids=all_uids)
print(f'Downloaded {len(objects)} objects.')

for subset in ['hard_set', 'easy_set', 'all_scored']:
    for obj in split[subset]:
        if obj['uid'] in objects:
            obj['glb_path'] = objects[obj['uid']]

with open(SPLIT_JSON, 'w') as f:
    json.dump(split, f, indent=2)
print('Updated dataset_split.json with local GLB paths.')

# Render dataset images
os.makedirs(os.path.join(RENDER_DIR, 'hard'), exist_ok=True)
os.makedirs(os.path.join(RENDER_DIR, 'easy'), exist_ok=True)

manifest = {'hard': {}, 'easy': {}}
H = W = 512

for subset, label in [('hard_set', 'hard'), ('easy_set', 'easy')]:
    for obj in split[subset]:
        uid, glb_path = obj['uid'], obj['glb_path']
        out_path = os.path.join(RENDER_DIR, label, f'{uid}.png')

        if os.path.exists(out_path):
            manifest[label][uid] = out_path
            continue

        if not os.path.exists(glb_path):
            print(f'  SKIP {uid[:8]}: GLB missing')
            continue

        print(f'  Rendering {uid[:8]} ({label})...')
        try:
            scene_or_mesh = trimesh.load(glb_path)
            if isinstance(scene_or_mesh, trimesh.Scene):
                meshes = [g for g in scene_or_mesh.geometry.values()
                          if isinstance(g, trimesh.Trimesh)]
                mesh = trimesh.util.concatenate(meshes) if meshes else None
            else:
                mesh = scene_or_mesh
            if mesh is None:
                continue

            mesh = trimesh.Trimesh(vertices=mesh.vertices, faces=mesh.faces, process=False)
            mesh.apply_translation(-mesh.centroid)
            ext = max(mesh.extents)
            if ext > 0:
                mesh.apply_scale(1.0 / ext)

            cam_dist = 1.9
            half_h = cam_dist * np.tan(np.radians(40.0) / 2)
            ys = np.linspace(half_h, -half_h, H)
            xs = np.linspace(-half_h, half_h, W)
            xx, yy = np.meshgrid(xs, ys)

            origins = np.zeros((H * W, 3), dtype=np.float64)
            origins[:, 0] = xx.ravel()
            origins[:, 1] = yy.ravel()
            origins[:, 2] = cam_dist
            directions = np.zeros((H * W, 3), dtype=np.float64)
            directions[:, 2] = -1.0

            intersector = trimesh.ray.ray_triangle.RayMeshIntersector(mesh)
            locations, index_ray, index_tri = intersector.intersects_location(
                ray_origins=origins, ray_directions=directions, multiple_hits=False)

            rgb = np.ones((H * W, 3), dtype=np.float32) * 255
            if len(locations) > 0:
                fn = mesh.face_normals[index_tri]
                d = np.clip(0.3 + 0.7 * np.abs(fn @ [0, 0, 1]).astype(np.float32), 0, 1)
                rgb[index_ray] = (d[:, None] * 255).astype(np.float32)

            img = Image.fromarray(rgb.reshape(H, W, 3).astype(np.uint8), 'RGB').convert('RGBA')
            alpha = np.ones((H, W), dtype=np.uint8) * 255
            if len(locations) > 0:
                hit = np.zeros(H * W, dtype=bool)
                hit[index_ray] = True
                alpha[~hit.reshape(H, W)] = 0
            else:
                alpha[:] = 0
            img.putalpha(Image.fromarray(alpha))
            img.save(out_path)
            manifest[label][uid] = out_path
        except Exception as e:
            print(f'  ERROR {uid[:8]}: {e}')

with open(RENDER_MANIFEST, 'w') as f:
    json.dump(manifest, f, indent=2)

print(f'\nRenders: hard={len(manifest["hard"])}, easy={len(manifest["easy"])}')
print('Dataset ready.')

## Cell 6 — Fine-tune TripoSR with back-face + silhouette loss

This is the main training cell. Takes ~5-10 min on an A100.

In [ ]:
import torch
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR

sys.path.insert(0, TRIPOSR_DIR)
from tsr.system import TSR
from tsr.utils import get_spherical_cameras, remove_background, resize_foreground

# ─── Hyperparameters (adjust as needed) ──────────────────────────────────────
LAMBDA_BACKFACE = 0.05
LAMBDA_SIL      = 1.0
LR              = 5e-5
N_EPOCHS        = 10
WARMUP_EPOCHS   = 3
N_VIEWS         = 2
IMG_SIZE        = 64
SAVE_EVERY      = 2
DEVICE          = 'cuda'

CHECKPOINT_DIR = os.path.join(BASE_DIR, 'checkpoints_backface_v2')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ─── Helper functions ─────────────────────────────────────────────────────────
def load_and_normalize_mesh(glb_path):
    s = trimesh.load(glb_path)
    if isinstance(s, trimesh.Scene):
        parts = [g for g in s.geometry.values() if isinstance(g, trimesh.Trimesh)]
        if not parts: raise ValueError('No meshes')
        m = trimesh.util.concatenate(parts)
    else:
        m = s
    m = trimesh.Trimesh(vertices=m.vertices, faces=m.faces, process=False)
    m.apply_translation(-m.centroid)
    ext = max(m.extents)
    if ext < 1e-8: raise ValueError('Degenerate')
    m.apply_scale(1.0 / ext)
    return m

def render_gt_rgb(mesh, rays_o, rays_d):
    H, W = rays_o.shape[:2]
    ro = rays_o.reshape(-1, 3).cpu().numpy().astype(np.float64)
    rd = rays_d.reshape(-1, 3).cpu().numpy().astype(np.float64)
    inter = trimesh.ray.ray_triangle.RayMeshIntersector(mesh)
    locs, idx_ray, idx_tri = inter.intersects_location(ray_origins=ro, ray_directions=rd, multiple_hits=False)
    rgb = np.ones((H * W, 3), dtype=np.float32)
    if len(locs) > 0:
        d = np.clip(0.3 + 0.7 * np.abs(mesh.face_normals[idx_tri] @ [0,0,1]).astype(np.float32), 0, 1)
        rgb[idx_ray] = d[:, None]
    return torch.from_numpy(rgb.reshape(H, W, 3)).to(DEVICE)

def render_gt_sil(mesh, rays_o, rays_d):
    H, W = rays_o.shape[:2]
    ro = rays_o.reshape(-1, 3).cpu().numpy().astype(np.float64)
    rd = rays_d.reshape(-1, 3).cpu().numpy().astype(np.float64)
    inter = trimesh.ray.ray_triangle.RayMeshIntersector(mesh)
    hits = inter.intersects_any(ray_origins=ro, ray_directions=rd)
    return torch.from_numpy(hits.astype(np.float32)).reshape(H, W).to(DEVICE)

def get_backface_cameras(cd=1.9, fov=40.0, h=64, w=64):
    ro, rd = get_spherical_cameras(n_views=1, elevation_deg=0.0, camera_distance=cd, fovy_deg=fov, height=h, width=w)
    rot = torch.tensor([[-1,0,0],[0,1,0],[0,0,-1]], dtype=ro.dtype)
    s = ro.shape
    return (ro.reshape(-1,3) @ rot.T).reshape(s), (rd.reshape(-1,3) @ rot.T).reshape(s)

def rgba_to_rgb_white_bg(image):
    if image.mode != 'RGBA': return image.convert('RGB')
    bg = Image.new('RGBA', image.size, (255,255,255,255))
    bg.paste(image, mask=image.split()[3])
    return bg.convert('RGB')

# ─── Load data ────────────────────────────────────────────────────────────────
with open(SPLIT_JSON) as f:
    split = json.load(f)

train_data = []
for subset, label in [('hard_set', 'hard'), ('easy_set', 'easy')]:
    for obj in split[subset]:
        uid = obj['uid']
        img_path = os.path.join(RENDER_DIR, label, f'{uid}.png')
        if os.path.exists(img_path) and os.path.exists(obj['glb_path']):
            train_data.append({'uid': uid, 'glb_path': obj['glb_path'], 'img_path': img_path, 'label': label})

print(f'Training on {len(train_data)} objects')

# ─── Load model ───────────────────────────────────────────────────────────────
model = TSR.from_pretrained('stabilityai/TripoSR', config_name='config.yaml', weight_name='model.ckpt')
model.renderer.set_chunk_size(0)
model.to(DEVICE)
model.train()

for p in model.image_tokenizer.parameters(): p.requires_grad = False
for p in model.tokenizer.parameters(): p.requires_grad = True
for p in model.backbone.parameters(): p.requires_grad = True
for p in model.post_processor.parameters(): p.requires_grad = True
for p in model.decoder.parameters(): p.requires_grad = True

n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {n_train:,}')

optimizer = Adam([
    {'params': list(model.backbone.parameters()), 'lr': LR * 0.01},
    {'params': list(model.tokenizer.parameters()), 'lr': LR * 0.01},
    {'params': list(model.post_processor.parameters()), 'lr': LR},
    {'params': list(model.decoder.parameters()), 'lr': LR},
])
scheduler = CosineAnnealingLR(optimizer, T_max=N_EPOCHS)
log = []

# ─── Training loop ────────────────────────────────────────────────────────────
for epoch in range(1, N_EPOCHS + 1):
    ramp = min(1.0, epoch / WARMUP_EPOCHS)
    e_rgb, e_sil, e_back, e_total = [], [], [], []

    for obj in train_data:
        uid = obj['uid']
        try:
            gt_mesh = load_and_normalize_mesh(obj['glb_path'])
        except Exception as e:
            print(f'  SKIP {uid[:8]}: {e}'); continue

        image = Image.open(obj['img_path']).convert('RGBA')
        image = remove_background(image)
        image = resize_foreground(image, 0.85)
        image = rgba_to_rgb_white_bg(image)

        optimizer.zero_grad()
        scene_codes = model([image], device=DEVICE)

        rays_o, rays_d = get_spherical_cameras(
            n_views=N_VIEWS, elevation_deg=0.0, camera_distance=1.9,
            fovy_deg=40.0, height=IMG_SIZE, width=IMG_SIZE)
        rays_o, rays_d = rays_o.to(DEVICE), rays_d.to(DEVICE)

        loss_rgb_v, loss_sil_v = [], []
        for v in range(N_VIEWS):
            pred_rgb, pred_opa = model.renderer(model.decoder, scene_codes[0], rays_o[v], rays_d[v])
            gt_r = render_gt_rgb(gt_mesh, rays_o[v], rays_d[v])
            gt_s = render_gt_sil(gt_mesh, rays_o[v], rays_d[v])
            loss_rgb_v.append(F.l1_loss(pred_rgb, gt_r))
            pm = pred_opa.squeeze(-1) if pred_opa.dim() > 2 else pred_opa
            loss_sil_v.append(F.binary_cross_entropy(pm.clamp(1e-6, 1-1e-6), gt_s))

        l_rgb = torch.stack(loss_rgb_v).mean()
        l_sil = torch.stack(loss_sil_v).mean()

        ro_b, rd_b = get_backface_cameras(h=IMG_SIZE, w=IMG_SIZE)
        ro_b, rd_b = ro_b.to(DEVICE), rd_b.to(DEVICE)
        pred_b, _ = model.renderer(model.decoder, scene_codes[0], ro_b[0], rd_b[0])
        gt_b = render_gt_rgb(gt_mesh, ro_b[0], rd_b[0])
        l_bf = F.l1_loss(pred_b, gt_b)

        l_total = l_rgb + LAMBDA_SIL * l_sil + ramp * LAMBDA_BACKFACE * l_bf
        l_total.backward()
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
        optimizer.step()

        e_rgb.append(l_rgb.item()); e_sil.append(l_sil.item())
        e_back.append(l_bf.item()); e_total.append(l_total.item())

    scheduler.step()
    mr = np.mean(e_rgb); ms = np.mean(e_sil); mb = np.mean(e_back); mt = np.mean(e_total)
    print(f'Epoch {epoch}/{N_EPOCHS} (ramp={ramp:.2f}) — rgb={mr:.4f}  sil={ms:.4f}  bf={mb:.4f}  total={mt:.4f}')
    log.append({'epoch': epoch, 'ramp': ramp, 'mean_rgb': float(mr), 'mean_sil': float(ms), 'mean_backface': float(mb), 'mean_total': float(mt)})

    if epoch % SAVE_EVERY == 0:
        torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, f'epoch_{epoch:03d}.ckpt'))

torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, 'final.ckpt'))
with open(os.path.join(CHECKPOINT_DIR, 'training_log.json'), 'w') as f:
    json.dump(log, f, indent=2)
print(f'\nTraining done. Checkpoint: {CHECKPOINT_DIR}/final.ckpt')

## Cell 7 — Run inference on all 20 objects

Uses adaptive thresholding to handle varying density scales after fine-tuning.

In [ ]:
OUTPUT_DIR = os.path.join(BASE_DIR, 'inference_outputs_backface_v2')
CHECKPOINT = os.path.join(CHECKPOINT_DIR, 'final.ckpt')
INF_MANIFEST = os.path.join(DATA_DIR, 'inference_manifest_backface_v2.json')

os.makedirs(os.path.join(OUTPUT_DIR, 'hard'), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'easy'), exist_ok=True)

# Reload model in eval mode
model_inf = TSR.from_pretrained('stabilityai/TripoSR', config_name='config.yaml', weight_name='model.ckpt')
ckpt = torch.load(CHECKPOINT, map_location='cpu')
model_inf.load_state_dict(ckpt)
model_inf.renderer.set_chunk_size(131072)
model_inf.to('cuda')
model_inf.eval()
print(f'Loaded checkpoint: {CHECKPOINT}\n')

with open(RENDER_MANIFEST) as f:
    manifest = json.load(f)

results = {}
for label in ['hard', 'easy']:
    print(f'Inference on {label} set...')
    for uid, img_path in manifest[label].items():
        out_path = os.path.join(OUTPUT_DIR, label, f'{uid}.obj')
        if os.path.exists(out_path):
            results[uid] = {'label': label, 'mesh_path': out_path}
            continue
        try:
            image = Image.open(img_path).convert('RGBA')
            image = remove_background(image)
            image = resize_foreground(image, 0.85)
            image = rgba_to_rgb_white_bg(image)
            with torch.no_grad():
                sc = model_inf([image], device='cuda')
            for thresh in [10.0, 5.0, 2.0, 1.0]:
                try:
                    meshes = model_inf.extract_mesh(sc, has_vertex_color=True, resolution=256, threshold=thresh)
                    if meshes[0].vertices.shape[0] > 0:
                        meshes[0].export(out_path)
                        results[uid] = {'label': label, 'mesh_path': out_path}
                        print(f'  {uid[:8]}: OK (thresh={thresh})')
                        break
                except: continue
            else:
                print(f'  {uid[:8]}: FAIL')
        except Exception as e:
            print(f'  {uid[:8]}: ERROR {e}')

with open(INF_MANIFEST, 'w') as f:
    json.dump(results, f, indent=2)

total = len(manifest.get('hard', {})) + len(manifest.get('easy', {}))
print(f'\nDone: {len(results)}/{total} objects.')

## Cell 8 — Compute CD and F-score

In [ ]:
from scipy.spatial import cKDTree

BASELINE_JSON = os.path.join(DATA_DIR, 'metrics_baseline.json')
OUTPUT_JSON = os.path.join(DATA_DIR, 'metrics_backface_v2.json')
N_SAMPLES = 10000
F_THRESH = 0.05

def sample_surface(path, n=N_SAMPLES):
    m = trimesh.load(path, force='mesh')
    if isinstance(m, trimesh.Scene): m = m.dump(concatenate=True)
    m.apply_translation(-m.centroid)
    m.apply_scale(1.0 / max(m.extents))
    pts, _ = trimesh.sample.sample_surface(m, n)
    return pts

def chamfer_dist(a, b):
    da, _ = cKDTree(b).query(a)
    db, _ = cKDTree(a).query(b)
    return float(np.mean(da**2) + np.mean(db**2))

def f_score_fn(a, b, t=F_THRESH):
    da, _ = cKDTree(b).query(a)
    db, _ = cKDTree(a).query(b)
    p, r = np.mean(da < t), np.mean(db < t)
    return float(2*p*r/(p+r)) if (p+r) > 0 else 0.0

with open(INF_MANIFEST) as f:
    inf_data = json.load(f)
with open(BASELINE_JSON) as f:
    baseline = json.load(f)

gt_paths = {}
for subset in ['hard_set', 'easy_set']:
    for obj in split[subset]:
        gt_paths[obj['uid']] = obj['glb_path']

bl_lookup = {}
for label in ['hard', 'easy']:
    for obj in baseline['per_object'][label]:
        bl_lookup[obj['uid']] = obj

results_cd = {'hard': [], 'easy': []}
scores = {k: {'cd': [], 'fs': []} for k in ['hard', 'easy']}

for uid, info in inf_data.items():
    label = info['label']
    gt = gt_paths.get(uid)
    if gt is None or not os.path.exists(info['mesh_path']): continue
    try:
        cd = chamfer_dist(sample_surface(info['mesh_path']), sample_surface(gt))
        fs = f_score_fn(sample_surface(info['mesh_path']), sample_surface(gt))
        scores[label]['cd'].append(cd)
        scores[label]['fs'].append(fs)
        bl = bl_lookup.get(uid, {})
        results_cd[label].append({'uid': uid, 'cd': cd, 'fs': fs,
                                   'baseline_cd': bl.get('chamfer_distance'),
                                   'baseline_fs': bl.get('f_score')})
        print(f'  {uid[:8]} ({label}): CD={cd:.4f}  F={fs:.4f}')
    except Exception as e:
        print(f'  {uid[:8]}: ERROR {e}')

summary = {}
for label in ['hard', 'easy']:
    summary[label] = {
        'mean_chamfer': float(np.mean(scores[label]['cd'])) if scores[label]['cd'] else None,
        'mean_f_score': float(np.mean(scores[label]['fs'])) if scores[label]['fs'] else None,
        'n': len(scores[label]['cd']),
    }

with open(OUTPUT_JSON, 'w') as f:
    json.dump({'summary': summary, 'per_object': results_cd}, f, indent=2)

bs = baseline['summary']
print(f'\n{"="*70}')
print(f'{"":20} {"Baseline":>12} {"BF v2":>12} {"Delta":>12}')
print(f'{"-"*70}')
for label in ['hard', 'easy']:
    bc, bf = bs[label]['mean_chamfer'], bs[label]['mean_f_score']
    c2, f2 = summary[label]['mean_chamfer'], summary[label]['mean_f_score']
    tag = label.capitalize()
    print(f'{tag+" CD":20} {bc:>12.4f} {c2:>12.4f} {c2-bc:>+12.4f}')
    print(f'{tag+" F-score":20} {bf:>12.4f} {f2:>12.4f} {f2-bf:>+12.4f}')
    if label == 'hard': print(f'{"-"*70}')
print(f'{"="*70}')

## Cell 9 — Compute PSNR, LPIPS, and render comparison images

In [ ]:
import lpips
from skimage.metrics import peak_signal_noise_ratio as compute_psnr

COMP_DIR = os.path.join(BASE_DIR, 'comparison_images')
os.makedirs(COMP_DIR, exist_ok=True)
H_img = W_img = 256

lpips_fn = lpips.LPIPS(net='alex')

def render_mesh_rc(mesh, H, W, cam_dist=1.9, fov_deg=40.0):
    half_h = cam_dist * np.tan(np.radians(fov_deg) / 2)
    ys = np.linspace(half_h, -half_h, H)
    xs = np.linspace(-half_h, half_h, W)
    xx, yy = np.meshgrid(xs, ys)
    origins = np.zeros((H*W, 3), dtype=np.float64)
    origins[:,0] = xx.ravel(); origins[:,1] = yy.ravel(); origins[:,2] = cam_dist
    dirs = np.zeros((H*W, 3), dtype=np.float64); dirs[:,2] = -1.0
    inter = trimesh.ray.ray_triangle.RayMeshIntersector(mesh)
    locs, ir, it = inter.intersects_location(ray_origins=origins, ray_directions=dirs, multiple_hits=False)
    rgb = np.ones((H*W, 3), dtype=np.float32) * 255
    if len(locs) > 0:
        d = np.clip(0.3 + 0.7*np.abs(mesh.face_normals[it] @ [0,0,1]).astype(np.float32), 0, 1)
        rgb[ir] = (d[:,None] * 255).astype(np.float32)
    return rgb.reshape(H, W, 3).astype(np.uint8)

def load_norm(path):
    m = trimesh.load(path, force='mesh')
    if isinstance(m, trimesh.Scene): m = m.dump(concatenate=True)
    m = trimesh.Trimesh(vertices=m.vertices, faces=m.faces, process=False)
    m.apply_translation(-m.centroid)
    ext = max(m.extents)
    if ext > 1e-8: m.apply_scale(1.0/ext)
    return m

all_psnr = {'hard': [], 'easy': []}
all_lpips = {'hard': [], 'easy': []}
img_results = {'hard': [], 'easy': []}

for uid, info in inf_data.items():
    label = info['label']
    gt = gt_paths.get(uid)
    if gt is None or not os.path.exists(info['mesh_path']) or not os.path.exists(gt): continue
    try:
        pred_img = render_mesh_rc(load_norm(info['mesh_path']), H_img, W_img)
        gt_img = render_mesh_rc(load_norm(gt), H_img, W_img)
        psnr_val = compute_psnr(gt_img, pred_img, data_range=255)
        pt = torch.from_numpy(pred_img).permute(2,0,1).float() / 127.5 - 1.0
        gt_t = torch.from_numpy(gt_img).permute(2,0,1).float() / 127.5 - 1.0
        with torch.no_grad():
            lp = lpips_fn(pt.unsqueeze(0), gt_t.unsqueeze(0)).item()
        all_psnr[label].append(psnr_val)
        all_lpips[label].append(lp)
        img_results[label].append({'uid': uid, 'psnr': psnr_val, 'lpips': lp})
        print(f'  {uid[:8]} ({label}): PSNR={psnr_val:.2f}  LPIPS={lp:.4f}')

        combined = np.zeros((H_img, W_img*2+10, 3), dtype=np.uint8)
        combined[:, :W_img] = gt_img
        combined[:, W_img+10:] = pred_img
        combined[:, W_img:W_img+10] = 128
        Image.fromarray(combined).save(os.path.join(COMP_DIR, f'{uid[:8]}_{label}.png'))
    except Exception as e:
        print(f'  {uid[:8]}: ERROR {e}')

print(f'\n{"="*65}')
print(f'IMAGE METRICS: PSNR (higher=better) and LPIPS (lower=better)')
print(f'{"="*65}')
print(f'{"":20} {"PSNR":>12} {"LPIPS":>12} {"N":>6}')
print(f'{"-"*65}')
for label in ['hard', 'easy']:
    if all_psnr[label]:
        print(f'{label.capitalize():20} {np.mean(all_psnr[label]):>12.2f} {np.mean(all_lpips[label]):>12.4f} {len(all_psnr[label]):>6}')
print(f'{"="*65}')

with open(os.path.join(DATA_DIR, 'image_metrics_v2.json'), 'w') as f:
    json.dump({'per_object': img_results,
               'summary': {l: {'mean_psnr': float(np.mean(all_psnr[l])) if all_psnr[l] else None,
                                'mean_lpips': float(np.mean(all_lpips[l])) if all_lpips[l] else None,
                                'n': len(all_psnr[l])} for l in ['hard', 'easy']}}, f, indent=2)

print(f'\nComparison images: {COMP_DIR}/')

## Cell 10 — Visualize comparison images

Display a few side-by-side comparisons (GT left, predicted right).

In [ ]:
import matplotlib.pyplot as plt

comp_files = sorted([f for f in os.listdir(COMP_DIR) if f.endswith('.png')])

n = min(6, len(comp_files))
fig, axes = plt.subplots(n, 1, figsize=(10, 3*n))
if n == 1: axes = [axes]

for i, fname in enumerate(comp_files[:n]):
    img = Image.open(os.path.join(COMP_DIR, fname))
    axes[i].imshow(img)
    axes[i].set_title(fname.replace('.png', '  (GT left | Predicted right)'), fontsize=11)
    axes[i].axis('off')

plt.tight_layout()
plt.show()